# 🛰 Bhubaneswar Change Detection — Analysis Notebook

**Satellite Land-Use Change Analysis · 2018–2026 · Odisha, India**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR_GITHUB_USERNAME/bhubaneswar-change-detection/blob/main/notebooks/bhubaneswar_analysis.ipynb)

This notebook:
1. Authenticates with Google Earth Engine
2. Computes Sentinel-2 composites for four 2-year periods (2018→2020, 2020→2022, 2022→2024, 2024→2026)
3. Generates swipeable before/after maps using `geemap`
4. Computes land-cover area statistics (vegetation, water, built-up)
5. Produces a cross-period comparison table and exports to CSV

**GEE Project**: `change-detection-494607`  
**Asset namespace**: `bhubaneswar_change_detection`

In [ ]:
# ── Install dependencies (Colab only) ─────────────────────────────────────────
import sys
if 'google.colab' in sys.modules:
    !pip install -q earthengine-api geemap pandas

In [ ]:
# ── Authenticate & initialise Earth Engine ─────────────────────────────────────
import ee, geemap, pandas as pd

try:
    ee.Initialize(project='change-detection-494607')
    print('[OK] Earth Engine initialised')
except Exception:
    ee.Authenticate()
    ee.Initialize(project='change-detection-494607')
    print('[OK] Earth Engine initialised after auth')

In [ ]:
# ── Study area & periods ───────────────────────────────────────────────────────

# Bhubaneswar Municipal Corporation bounding box [W, S, E, N]
BBSR_BOUNDS = [85.7200, 20.1200, 85.9800, 20.4000]
ROI = ee.Geometry.Rectangle(BBSR_BOUNDS)

PERIODS = [
    {'year1': 2018, 'year2': 2020, 'label': '2018 → 2020', 'ctx': 'Post-Cyclone Titli recovery'},
    {'year1': 2020, 'year2': 2022, 'label': '2020 → 2022', 'ctx': 'COVID lockdown greening'},
    {'year1': 2022, 'year2': 2024, 'label': '2022 → 2024', 'ctx': 'Smart City Phase-2'},
    {'year1': 2024, 'year2': 2026, 'label': '2024 → 2026', 'ctx': 'Metro expansion'},
]

CHANGE_PALETTE = ['#ef4444','#f97316','#facc15','#22c55e','#3b82f6','#06b6d4']
CENTER = [20.2961, 85.8245]   # (lat, lon)

print(f'Study area: {BBSR_BOUNDS}')
print(f'Periods:    {[p["label"] for p in PERIODS]}')

In [ ]:
# ── Helper functions ───────────────────────────────────────────────────────────

def mask_s2_clouds(image):
    qa = image.select('QA60')
    mask = (qa.bitwiseAnd(1 << 10).eq(0)
              .And(qa.bitwiseAnd(1 << 11).eq(0)))
    return image.updateMask(mask)

def s2_composite(year):
    """Feb–March median composite (peak Rabi season)."""
    return (
        ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
        .filterDate(f'{year}-02-01', f'{year}-03-31')
        .filterBounds(ROI)
        .map(mask_s2_clouds)
        .median()
        .select(['B3','B4','B8','B11'])
        .divide(10000)
        .clip(ROI)
    )

def classify(img):
    """Return 3-class image: 1=vegetation, 2=water, 3=builtup."""
    ndvi = img.normalizedDifference(['B8','B4'])
    ndwi = img.normalizedDifference(['B3','B8'])
    ndbi = img.normalizedDifference(['B11','B8'])
    bui  = ndbi.subtract(ndvi)
    return (
        ee.Image(0)
        .where(ndvi.gt(0.30).And(ndwi.lte(0.10)), 1)  # vegetation
        .where(ndwi.gt(0.10),                      2)  # water
        .where(bui.gt(0.0).And(ndwi.lte(0.05)),   3)  # built-up
    )

def change_map(year1, year2):
    """6-class change image."""
    c1, c2 = classify(s2_composite(year1)), classify(s2_composite(year2))
    chg = (
        ee.Image(0)
        .where(c1.neq(3).And(c2.eq(3)),  1)   # built-up gain
        .where(c1.eq(3).And(c2.neq(3)),  2)   # built-up loss
        .where(c1.eq(1).And(c2.neq(1)),  3)   # veg loss
        .where(c1.neq(1).And(c2.eq(1)),  4)   # veg gain
        .where(c1.neq(2).And(c2.eq(2)),  5)   # water gain
        .where(c1.eq(2).And(c2.neq(2)),  6)   # water recession
    )
    return chg.updateMask(chg.gt(0))

def area_stats(year):
    """Return vegetation / water / built-up area in km²."""
    img  = s2_composite(year)
    ndvi = img.normalizedDifference(['B8','B4'])
    ndwi = img.normalizedDifference(['B3','B8'])
    bui  = img.normalizedDifference(['B11','B8']).subtract(ndvi)
    px   = ee.Image.pixelArea().divide(1e6)
    stacked = (
        ndvi.gt(0.30).And(ndwi.lte(0.10)).multiply(px).rename('vegetation')
        .addBands(ndwi.gt(0.10).multiply(px).rename('water'))
        .addBands(bui.gt(0.0).And(ndwi.lte(0.05)).multiply(px).rename('builtup'))
    )
    res = stacked.reduceRegion(
        reducer=ee.Reducer.sum(), geometry=ROI,
        scale=60, bestEffort=False, maxPixels=1e8, tileScale=4
    ).getInfo()
    return {k: round(v or 0, 2) for k, v in res.items()}

print('Helper functions defined.')

## Period 1: 2018 → 2020
*Post-Cyclone Titli recovery & early Smart City phase*

In [ ]:
# ── Swipe map: RGB comparison 2018 vs 2020 ────────────────────────────────────
m1 = geemap.Map(center=CENTER, zoom=12)
rgb_vis = {'min': 0.02, 'max': 0.28, 'bands': ['B4','B3','B2']}

img_2018 = s2_composite(2018).select(['B4','B3','B2'])
img_2020 = s2_composite(2020).select(['B4','B3','B2'])

m1.split_map(
    left_layer  = geemap.ee_tile_layer(img_2018, rgb_vis, 'RGB 2018'),
    right_layer = geemap.ee_tile_layer(img_2020, rgb_vis, 'RGB 2020'),
    left_label  = '2018',
    right_label = '2020',
)
m1

In [ ]:
# ── Change map 2018→2020 ───────────────────────────────────────────────────────
m1b = geemap.Map(center=CENTER, zoom=12)
m1b.addLayer(img_2018, rgb_vis, 'RGB 2018 (base)')
m1b.addLayer(change_map(2018, 2020),
             {'min':1,'max':6,'palette':CHANGE_PALETTE}, 'Change 2018→2020')
m1b.add_legend(title='Change Class',
               labels=['Built-up Gain','Built-up Loss','Veg Loss',
                       'Veg Gain','Water Gain','Water Recession'],
               colors=CHANGE_PALETTE)
m1b

In [ ]:
# ── Area statistics 2018 vs 2020 ───────────────────────────────────────────────
s18 = area_stats(2018)
s20 = area_stats(2020)
print('2018:', s18)
print('2020:', s20)

## Period 2: 2020 → 2022
*COVID-19 lockdown greening & infrastructure resumption*

In [ ]:
m2 = geemap.Map(center=CENTER, zoom=12)
img_2022 = s2_composite(2022).select(['B4','B3','B2'])
m2.split_map(
    left_layer  = geemap.ee_tile_layer(img_2020, rgb_vis, 'RGB 2020'),
    right_layer = geemap.ee_tile_layer(img_2022, rgb_vis, 'RGB 2022'),
    left_label='2020', right_label='2022',
)
m2

In [ ]:
m2b = geemap.Map(center=CENTER, zoom=12)
m2b.addLayer(img_2020, rgb_vis, 'RGB 2020 (base)')
m2b.addLayer(change_map(2020,2022), {'min':1,'max':6,'palette':CHANGE_PALETTE}, 'Change 2020→2022')
m2b

## Period 3: 2022 → 2024
*Smart City Phase-2 & metro corridor development*

In [ ]:
m3 = geemap.Map(center=CENTER, zoom=12)
img_2024 = s2_composite(2024).select(['B4','B3','B2'])
m3.split_map(
    left_layer  = geemap.ee_tile_layer(img_2022, rgb_vis, 'RGB 2022'),
    right_layer = geemap.ee_tile_layer(img_2024, rgb_vis, 'RGB 2024'),
    left_label='2022', right_label='2024',
)
m3

In [ ]:
m3b = geemap.Map(center=CENTER, zoom=12)
m3b.addLayer(img_2022, rgb_vis, 'RGB 2022 (base)')
m3b.addLayer(change_map(2022,2024), {'min':1,'max':6,'palette':CHANGE_PALETTE}, 'Change 2022→2024')
m3b

## Period 4: 2024 → 2026
*Metro expansion & recent urban growth*

In [ ]:
m4 = geemap.Map(center=CENTER, zoom=12)
img_2026 = s2_composite(2026).select(['B4','B3','B2'])
m4.split_map(
    left_layer  = geemap.ee_tile_layer(img_2024, rgb_vis, 'RGB 2024'),
    right_layer = geemap.ee_tile_layer(img_2026, rgb_vis, 'RGB 2026'),
    left_label='2024', right_label='2026',
)
m4

In [ ]:
m4b = geemap.Map(center=CENTER, zoom=12)
m4b.addLayer(img_2024, rgb_vis, 'RGB 2024 (base)')
m4b.addLayer(change_map(2024,2026), {'min':1,'max':6,'palette':CHANGE_PALETTE}, 'Change 2024→2026')
m4b

## Summary: Cross-Period Statistics

In [ ]:
# ── Compute all area stats ─────────────────────────────────────────────────────
print('Computing area stats for all years — this may take 2–5 minutes…')

all_years = [2018, 2020, 2022, 2024, 2026]
stats_by_year = {y: area_stats(y) for y in all_years}
print('Done.')
for y, s in stats_by_year.items():
    print(f'{y}: {s}')

In [ ]:
# ── Cross-period comparison table ──────────────────────────────────────────────
rows = []
for p in PERIODS:
    y1, y2 = p['year1'], p['year2']
    s1, s2 = stats_by_year[y1], stats_by_year[y2]
    for cls in ['vegetation','water','builtup']:
        v1, v2 = s1[cls], s2[cls]
        delta  = round(v2 - v1, 2)
        pct    = round((v2 - v1) / v1 * 100, 1) if v1 > 0 else None
        rows.append({
            'Period': p['label'],
            'Class':  cls.capitalize(),
            f'{y1} (km²)': v1,
            f'{y2} (km²)': v2,
            'Delta (km²)': delta,
            'Change (%)': pct,
        })

df = pd.DataFrame(rows)
df

In [ ]:
# ── Export to CSV ──────────────────────────────────────────────────────────────
df.to_csv('bhubaneswar_lulc_change_2018_2026.csv', index=False)
print('Exported: bhubaneswar_lulc_change_2018_2026.csv')

---
*Notebook · Airawat Research Foundation · GEE Project: change-detection-494607 · Asset namespace: bhubaneswar_change_detection*